In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, f1_score

Stratified K-Fold cross-validation preserves class proportions across folds. 
This is important for Speech Emotion Recognition because some emotions may 
appear less frequently than others.

Cross-validation provides a more reliable estimate of model generalization 
than a single train/test split.

In [7]:
X_train_sc = np.load('../outputs/X_train_sc.npy')
X_test_sc  = np.load('../outputs/X_test_sc.npy')
y_train    = np.load('../outputs/y_train.npy')
y_test     = np.load('../outputs/y_test.npy')
feat_cols  = pd.read_csv('../outputs/feature_names.csv').iloc[:, 0].tolist()

In [8]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

macro_f1 = make_scorer(f1_score, average="macro")

In [9]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "Linear SVM": SVC(kernel="linear", C=10),
    "RBF SVM": SVC(kernel="rbf", C=10, gamma=1/X_train_sc.shape[1]),
    "Decision Tree": DecisionTreeClassifier(max_depth=8),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

cv_results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train_sc,
        y_train,
        cv=skf,
        scoring=macro_f1,
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Mean Macro F1": scores.mean(),
        "Std": scores.std()
    })

cv_df = pd.DataFrame(cv_results)
cv_df

,Model,Mean Macro F1,Std
0,Logistic Regression,0.640918,0.025460
1,Linear SVM,0.603119,0.006105
2,RBF SVM,0.702696,0.017431
3,Decision Tree,0.342786,0.021099
4,Random Forest,0.584933,0.035468


MiniLearn